# Stage 2 Support: Local Pre-training Validation

This notebook records local forward/backward checks, smoke training, a 128-example overfit check, and a one-epoch timing probe. It provides supporting validation for the training recipe. The official 75-epoch runs are recorded in `01_train_models_standard_colab.ipynb` and `01_train_models_augmix_colab.ipynb`; their final prediction caches are recorded in `02_cache_model_outputs_colab.ipynb`.

Validation uses the CIFAR-10 training partition. The fixed calibration/evaluation indices for seeds 42, 43, and 44 are not used to fit the model or select a checkpoint.

The recipe evaluated here uses 75 epochs, batch size 64, SGD with learning rate 0.05, momentum 0.9, weight decay 0.0005, and a cosine learning-rate schedule. The one-epoch probe below was used to assess runtime before the official Colab training.

## 1. Imports, reproducibility, and device

This section sets fixed random seeds and chooses one device in the order CUDA, Intel XPU, then CPU. We print the software versions and accelerator details so the run is reproducible and the selected hardware is explicit.

In [1]:
import csv
import hashlib
import json
import platform
import random
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "data" / "raw"
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
RESULTS_DIR = PROJECT_ROOT / "results" / "resnet18_standard"
CACHE_DIR = PROJECT_ROOT / "cached_outputs"

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch, "xpu") and torch.xpu.is_available():
    device = torch.device("xpu")
else:
    device = torch.device("cpu")

if device.type == "cuda":
    device_name = torch.cuda.get_device_name(device)
elif device.type == "xpu":
    device_name = torch.xpu.get_device_name(device)
else:
    device_name = platform.processor() or "CPU"

print(f"Python: {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Intel XPU available: {hasattr(torch, 'xpu') and torch.xpu.is_available()}")
print(f"Selected device: {device} ({device_name})")
if hasattr(torch.backends, "cudnn"):
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

sys.path.insert(0, str(PROJECT_ROOT))
from src.models import resnet18_cifar

Python: 3.12.10
PyTorch: 2.14.1+cu126
torchvision: 0.29.1+cu126
CUDA available: True
Intel XPU available: False
Selected device: cuda (NVIDIA GeForce MX330)


## 2. CIFAR-10 data and training configuration

The model sees the original 32×32 RGB images. Training augmentation uses random crop with four-pixel padding and horizontal flips. Both training and clean-test inputs use channel-wise CIFAR-10 normalization:

\[
\mu=(0.4914,\;0.4822,\;0.4465),\qquad
\sigma=(0.2470,\;0.2435,\;0.2616).
\]

The official 50,000-example training set is the only source used for fitting. The clean test set is loaded for the later, final evaluation pass only.

In [2]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
evaluation_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])
plain_train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
])

train_dataset = CIFAR10(root=str(DATA_ROOT), train=True, download=False, transform=train_transform)
test_dataset = CIFAR10(root=str(DATA_ROOT), train=False, download=False, transform=evaluation_transform)
plain_train_dataset = CIFAR10(root=str(DATA_ROOT), train=True, download=False, transform=plain_train_transform)

TRAINING_CONFIG = {
    "model": "CIFAR-appropriate ResNet-18",
    "seed": SEED,
    "epochs": 75,
    "batch_size": 64,
    "optimizer": "SGD",
    "learning_rate": 0.05,
    "momentum": 0.9,
    "weight_decay": 0.0005,
    "scheduler": "CosineAnnealingLR",
    "scheduler_t_max": 75,
    "scheduler_eta_min": 0.0,
    "train_augmentation": ["RandomCrop(32, padding=4)", "RandomHorizontalFlip"],
    "normalization_mean": list(CIFAR_MEAN),
    "normalization_std": list(CIFAR_STD),
    "num_workers": 0,
}
print(f"Training examples: {len(train_dataset):,}")
print(f"Clean test examples: {len(test_dataset):,}")
print(f"Image tensor shape: {train_dataset[0][0].shape}")
print(f"Normalization mean: {CIFAR_MEAN}")
print(f"Normalization std:  {CIFAR_STD}")
print("Optimizer/scheduler configuration:")
print(json.dumps({k: TRAINING_CONFIG[k] for k in [
    "epochs", "batch_size", "optimizer", "learning_rate", "momentum",
    "weight_decay", "scheduler", "scheduler_t_max", "scheduler_eta_min"
]}, indent=2))
assert len(train_dataset) == 50_000 and len(test_dataset) == 10_000
assert train_dataset[0][0].shape == (3, 32, 32)
assert np.isfinite(train_dataset[0][0].numpy()).all()

train_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAINING_CONFIG["batch_size"],
    shuffle=True,
    num_workers=TRAINING_CONFIG["num_workers"],
    pin_memory=(device.type == "cuda"),
    generator=train_generator,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=TRAINING_CONFIG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

Training examples: 50,000
Clean test examples: 10,000
Image tensor shape: torch.Size([3, 32, 32])
Normalization mean: (0.4914, 0.4822, 0.4465)
Normalization std:  (0.247, 0.2435, 0.2616)
Optimizer/scheduler configuration:
{
  "epochs": 75,
  "batch_size": 64,
  "optimizer": "SGD",
  "learning_rate": 0.05,
  "momentum": 0.9,
  "weight_decay": 0.0005,
  "scheduler": "CosineAnnealingLR",
  "scheduler_t_max": 75,
  "scheduler_eta_min": 0.0
}


## 3. Architecture and tensor-shape check

Before optimization, inspect the CIFAR stem, residual-stage layout, parameter count, and logits shape. For a batch of 64 images, the network output must have shape 64×10.

In [3]:
model = resnet18_cifar(num_classes=10).to(device)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("Architecture: ResNet-18")
print(f"Stem: {model.conv1}; max-pool: {model.maxpool}")
print("Residual stages: layer1–layer4, 2 basic blocks per stage, widths 64/128/256/512")
print(f"Classifier: {model.fc}")
print(f"Trainable parameters: {parameter_count:,}")

shape_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(SEED + 100),
)
example_images, example_labels = next(iter(shape_loader))
example_images = example_images.to(device)
example_labels = example_labels.to(device)
with torch.no_grad():
    example_logits = model(example_images)
print(f"Input shape: {tuple(example_images.shape)}")
print(f"Label shape: {tuple(example_labels.shape)}")
print(f"Output logits shape: {tuple(example_logits.shape)}")
print(f"Model device: {next(model.parameters()).device}")
assert example_images.shape == (64, 3, 32, 32)
assert example_logits.shape == (64, 10)
model_device = next(model.parameters()).device
assert model_device.type == device.type
assert device.index is None or model_device.index == device.index
del example_images, example_labels, example_logits, shape_loader

Architecture: ResNet-18
Stem: Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False); max-pool: Identity()
Residual stages: layer1–layer4, 2 basic blocks per stage, widths 64/128/256/512
Classifier: Linear(in_features=512, out_features=10, bias=True)
Trainable parameters: 11,173,962


Input shape: (64, 3, 32, 32)
Label shape: (64,)
Output logits shape: (64, 10)
Model device: cuda:0


## 4. Two-batch forward/backward sanity check

Run two optimizer steps on batches from the CIFAR-10 training set. Verify finite cross-entropy, finite gradients, and an actual parameter update. This checks data-to-model dimensions, loss calculation, backpropagation, and device placement.

In [4]:
sanity_model = resnet18_cifar(num_classes=10).to(device)
sanity_optimizer = torch.optim.SGD(
    sanity_model.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    momentum=TRAINING_CONFIG["momentum"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
)
sanity_criterion = nn.CrossEntropyLoss()
sanity_generator = torch.Generator().manual_seed(SEED + 1)
sanity_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=0,
    pin_memory=(device.type == "cuda"), generator=sanity_generator,
)
weight_before = sanity_model.fc.weight.detach().clone()
two_batch_losses = []
for batch_number, (images, labels) in enumerate(sanity_loader):
    if batch_number == 2:
        break
    images, labels = images.to(device), labels.to(device)
    sanity_optimizer.zero_grad(set_to_none=True)
    logits = sanity_model(images)
    loss = sanity_criterion(logits, labels)
    assert torch.isfinite(loss), f"Non-finite loss in batch {batch_number + 1}"
    loss.backward()
    assert all(
        parameter.grad is None or torch.isfinite(parameter.grad).all()
        for parameter in sanity_model.parameters()
    ), f"Non-finite gradient in batch {batch_number + 1}"
    sanity_optimizer.step()
    two_batch_losses.append(float(loss.detach().cpu()))
assert len(two_batch_losses) == 2
assert not torch.equal(weight_before, sanity_model.fc.weight.detach())
print(f"Two-batch losses: {[round(value, 4) for value in two_batch_losses]}")
print("Finite loss, finite gradients, and parameter update: PASS")
del sanity_model, sanity_optimizer, sanity_loader
if device.type == "cuda":
    torch.cuda.empty_cache()

Two-batch losses: [2.4282, 2.4897]
Finite loss, finite gradients, and parameter update: PASS


## 5. Three-batch training smoke test

A fresh model runs three training batches (192 examples) with the planned optimizer. Losses should remain finite and at least one parameter must change. This is a short end-to-end training-path check; it is not used as the final model.

In [5]:
smoke_model = resnet18_cifar(num_classes=10).to(device)
smoke_optimizer = torch.optim.SGD(
    smoke_model.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    momentum=TRAINING_CONFIG["momentum"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
)
smoke_criterion = nn.CrossEntropyLoss()
smoke_generator = torch.Generator().manual_seed(SEED + 2)
smoke_loader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=0,
    pin_memory=(device.type == "cuda"), generator=smoke_generator,
)
smoke_weight_before = smoke_model.fc.weight.detach().clone()
smoke_losses = []
for batch_number, (images, labels) in enumerate(smoke_loader):
    if batch_number == 3:
        break
    images, labels = images.to(device), labels.to(device)
    smoke_optimizer.zero_grad(set_to_none=True)
    logits = smoke_model(images)
    loss = smoke_criterion(logits, labels)
    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite smoke-test loss at batch {batch_number + 1}")
    loss.backward()
    smoke_optimizer.step()
    smoke_losses.append(float(loss.detach().cpu()))
assert len(smoke_losses) == 3
assert not torch.equal(smoke_weight_before, smoke_model.fc.weight.detach())
print(f"Three-batch smoke losses: {[round(value, 4) for value in smoke_losses]}")
print("All losses finite; classifier weights updated: PASS")
del smoke_model, smoke_optimizer, smoke_loader
if device.type == "cuda":
    torch.cuda.empty_cache()

Three-batch smoke losses: [2.5014, 2.6156, 2.8983]
All losses finite; classifier weights updated: PASS


## 6. Small overfitting sanity check

Train a fresh model repeatedly on 128 fixed CIFAR-10 training images without random augmentation. This check should reduce the average training loss, showing that the model can fit a tiny subset. It uses training examples only and does not touch any calibration or evaluation split.

In [6]:
overfit_indices = np.arange(128, dtype=np.int64)
overfit_dataset = Subset(plain_train_dataset, overfit_indices.tolist())
overfit_generator = torch.Generator().manual_seed(SEED + 3)
overfit_loader = DataLoader(
    overfit_dataset, batch_size=64, shuffle=True, num_workers=0,
    pin_memory=(device.type == "cuda"), generator=overfit_generator,
)
overfit_model = resnet18_cifar(num_classes=10).to(device)
overfit_optimizer = torch.optim.SGD(
    overfit_model.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    momentum=TRAINING_CONFIG["momentum"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
)
overfit_criterion = nn.CrossEntropyLoss()
overfit_epoch_losses = []
for epoch in range(20):
    overfit_model.train()
    loss_sum = 0.0
    count = 0
    for images, labels in overfit_loader:
        images, labels = images.to(device), labels.to(device)
        overfit_optimizer.zero_grad(set_to_none=True)
        logits = overfit_model(images)
        loss = overfit_criterion(logits, labels)
        loss.backward()
        overfit_optimizer.step()
        loss_sum += float(loss.detach().cpu()) * len(labels)
        count += len(labels)
    overfit_epoch_losses.append(loss_sum / count)
print(f"128-image subset loss, epoch 1:  {overfit_epoch_losses[0]:.4f}")
print(f"128-image subset loss, epoch 20: {overfit_epoch_losses[-1]:.4f}")
print(f"Loss ratio (last / first): {overfit_epoch_losses[-1] / overfit_epoch_losses[0]:.3f}")
assert np.isfinite(overfit_epoch_losses).all()
assert overfit_epoch_losses[-1] < overfit_epoch_losses[0], (
    "Tiny-subset loss did not decrease; inspect this before timing a full epoch."
)
print("Small-subset overfitting check: PASS")
del overfit_model, overfit_optimizer, overfit_loader, overfit_dataset
if device.type == "cuda":
    torch.cuda.empty_cache()

128-image subset loss, epoch 1:  2.4100
128-image subset loss, epoch 20: 0.3843
Loss ratio (last / first): 0.159
Small-subset overfitting check: PASS


## 7. One-epoch runtime probe

This probe trains a fresh model for exactly one epoch over all 50,000 CIFAR-10 training examples. It saves no checkpoint and uses no calibration/evaluation data. Record the observed duration and estimate the 75-epoch training time before deciding whether to launch the full run.

In [7]:
probe_generator = torch.Generator().manual_seed(SEED + 4)
probe_loader = DataLoader(
    train_dataset, batch_size=TRAINING_CONFIG["batch_size"], shuffle=True, num_workers=0,
    pin_memory=(device.type == "cuda"), generator=probe_generator,
)
probe_model = resnet18_cifar(num_classes=10).to(device)
probe_optimizer = torch.optim.SGD(
    probe_model.parameters(),
    lr=TRAINING_CONFIG["learning_rate"],
    momentum=TRAINING_CONFIG["momentum"],
    weight_decay=TRAINING_CONFIG["weight_decay"],
)
probe_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    probe_optimizer,
    T_max=TRAINING_CONFIG["scheduler_t_max"],
    eta_min=TRAINING_CONFIG["scheduler_eta_min"],
)
probe_criterion = nn.CrossEntropyLoss()
probe_model.train()
probe_start = time.perf_counter()
probe_loss_sum = 0.0
probe_count = 0
for images, labels in probe_loader:
    images, labels = images.to(device), labels.to(device)
    probe_optimizer.zero_grad(set_to_none=True)
    logits = probe_model(images)
    loss = probe_criterion(logits, labels)
    if not torch.isfinite(loss):
        raise RuntimeError("Non-finite loss during the one-epoch runtime probe.")
    loss.backward()
    probe_optimizer.step()
    probe_loss_sum += float(loss.detach().cpu()) * len(labels)
    probe_count += len(labels)
probe_scheduler.step()
if device.type == "cuda":
    torch.cuda.synchronize()
probe_epoch_seconds = time.perf_counter() - probe_start
probe_average_loss = probe_loss_sum / probe_count
estimated_75_seconds = probe_epoch_seconds * TRAINING_CONFIG["epochs"]
print(f"Probe examples processed: {probe_count:,}")
print(f"One-epoch mean training loss: {probe_average_loss:.4f}")
print(f"One-epoch elapsed time: {probe_epoch_seconds / 60:.2f} minutes")
print(f"Linear 75-epoch estimate: {estimated_75_seconds / 3600:.2f} hours")
print("The probe model will be discarded; no checkpoint was written.")
del probe_model, probe_optimizer, probe_scheduler, probe_loader
if device.type == "cuda":
    torch.cuda.empty_cache()

Probe examples processed: 50,000
One-epoch mean training loss: 1.7778
One-epoch elapsed time: 17.10 minutes
Linear 75-epoch estimate: 21.38 hours
The probe model will be discarded; no checkpoint was written.


## 8. Optional local training path

The local full-training block is disabled by default and was not used for either official 75-epoch model. The official training records are in the two Colab notebooks, and shared model predictions are in the Stage 4 cache bundle.

If explicitly enabled, this optional block fits on the 50,000 training images and then saves a one-pass clean-test prediction cache. Its outputs are separate from the official Colab model artifacts. Calibration/evaluation data are not used for fitting or checkpoint selection.

In [8]:
RUN_FULL_TRAINING = False  # Leave disabled until the one-epoch runtime has been reviewed.

def checkpoint_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as checkpoint_file:
        for chunk in iter(lambda: checkpoint_file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def save_loss_history(history, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(output_file, fieldnames=["epoch", "mean_train_loss", "learning_rate"])
        writer.writeheader()
        writer.writerows(history)

def train_full_model():
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_path = CHECKPOINT_DIR / "resnet18_standard_cifar_seed42.pt"
    config_path = RESULTS_DIR / "training_config.json"
    history_path = RESULTS_DIR / "training_loss.csv"

    if config_path.exists():
        existing_config = json.loads(config_path.read_text(encoding="utf-8"))
        if existing_config != TRAINING_CONFIG:
            raise FileExistsError(
                f"Existing training config differs from this notebook: {config_path}. "
                "Preserve the previous run and choose a separate output name before proceeding."
            )
    else:
        config_path.write_text(json.dumps(TRAINING_CONFIG, indent=2), encoding="utf-8")

    model = resnet18_cifar(num_classes=10).to(device)
    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=TRAINING_CONFIG["learning_rate"],
        momentum=TRAINING_CONFIG["momentum"],
        weight_decay=TRAINING_CONFIG["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=TRAINING_CONFIG["scheduler_t_max"],
        eta_min=TRAINING_CONFIG["scheduler_eta_min"],
    )
    criterion = nn.CrossEntropyLoss()
    history = []
    start_epoch = 0

    if checkpoint_path.exists():
        state = torch.load(checkpoint_path, map_location=device, weights_only=False)
        if state["config"] != TRAINING_CONFIG:
            raise ValueError("Checkpoint configuration does not match this notebook.")
        model.load_state_dict(state["model_state_dict"])
        optimizer.load_state_dict(state["optimizer_state_dict"])
        scheduler.load_state_dict(state["scheduler_state_dict"])
        history = state["history"]
        start_epoch = state["epoch"]
        if start_epoch >= TRAINING_CONFIG["epochs"]:
            print(f"Reusing completed checkpoint at epoch {start_epoch}: {checkpoint_path}")
        else:
            random.setstate(state["python_rng_state"])
            np.random.set_state(state["numpy_rng_state"])
            torch.set_rng_state(state["torch_rng_state"])
            if device.type == "cuda" and state["cuda_rng_state"] is not None:
                torch.cuda.set_rng_state_all(state["cuda_rng_state"])
            train_loader.generator.set_state(state["loader_generator_state"])
            print(f"Resuming matching checkpoint from epoch {start_epoch}.")
    else:
        if history_path.exists():
            raise FileExistsError(
                f"Loss history exists without its checkpoint: {history_path}. "
                "Inspect and preserve these files before starting a new run."
            )

    for epoch in range(start_epoch, TRAINING_CONFIG["epochs"]):
        model.train()
        epoch_start = time.perf_counter()
        loss_sum = 0.0
        count = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            if not torch.isfinite(loss):
                raise RuntimeError(f"Non-finite training loss at epoch {epoch + 1}.")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach().cpu()) * len(labels)
            count += len(labels)
        scheduler.step()
        if device.type == "cuda":
            torch.cuda.synchronize()
        epoch_loss = loss_sum / count
        row = {
            "epoch": epoch + 1,
            "mean_train_loss": epoch_loss,
            "learning_rate": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        save_loss_history(history, history_path)

        state = {
            "epoch": epoch + 1,
            "config": TRAINING_CONFIG,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "python_rng_state": random.getstate(),
            "numpy_rng_state": np.random.get_state(),
            "torch_rng_state": torch.get_rng_state(),
            "cuda_rng_state": torch.cuda.get_rng_state_all() if device.type == "cuda" else None,
            "loader_generator_state": train_loader.generator.get_state(),
        }
        temporary_path = checkpoint_path.with_suffix(".tmp")
        torch.save(state, temporary_path)
        temporary_path.replace(checkpoint_path)
        epoch_seconds = time.perf_counter() - epoch_start
        print(
            f"Epoch {epoch + 1:02d}/{TRAINING_CONFIG['epochs']} | "
            f"mean train loss {epoch_loss:.4f} | "
            f"LR {optimizer.param_groups[0]['lr']:.6f} | "
            f"{epoch_seconds / 60:.2f} min"
        )

    return model, checkpoint_path

def cache_and_evaluate_clean_test(model, checkpoint_path):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = CACHE_DIR / "resnet18_standard_clean_test.npz"
    model_hash = checkpoint_sha256(checkpoint_path)

    if cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as saved:
            cached_hash = str(saved["model_sha256"].item())
            if cached_hash != model_hash:
                raise FileExistsError(
                    f"Existing clean-test cache belongs to a different checkpoint: {cache_path}. "
                    "Preserve it and choose a separate cache filename."
                )
            logits_array = saved["logits"].copy()
            probabilities_array = saved["probabilities"].copy()
            labels_array = saved["labels"].copy()
            index_array = saved["image_indices"].copy()
        print(f"Reusing clean-test cache for the current fixed checkpoint: {cache_path}")
    else:
        model.eval()
        logits_batches = []
        labels_batches = []
        with torch.inference_mode():
            for images, labels in test_loader:
                logits_batches.append(model(images.to(device)).cpu().numpy())
                labels_batches.append(labels.numpy())
        logits_array = np.concatenate(logits_batches, axis=0)
        labels_array = np.concatenate(labels_batches, axis=0).astype(np.int64)
        probabilities_array = torch.softmax(torch.from_numpy(logits_array), dim=1).numpy()
        index_array = np.arange(len(test_dataset), dtype=np.int64)
        np.savez_compressed(
            cache_path,
            image_indices=index_array,
            labels=labels_array,
            logits=logits_array,
            probabilities=probabilities_array,
            condition=np.asarray("clean"),
            severity=np.asarray(0, dtype=np.int64),
            model_sha256=np.asarray(model_hash),
        )
        print(f"Saved one-pass clean-test logits and probabilities: {cache_path}")

    assert logits_array.shape == (10_000, 10)
    assert probabilities_array.shape == (10_000, 10)
    assert labels_array.shape == (10_000,)
    assert np.array_equal(index_array, np.arange(10_000))
    split_accuracies = []
    for split_seed in (42, 43, 44):
        split_path = SPLIT_DIR / f"clean_test_split_seed{split_seed}.npz"
        with np.load(split_path, allow_pickle=False) as split:
            evaluation_indices = split["evaluation_indices"].astype(np.int64)
            calibration_indices = split["calibration_indices"].astype(np.int64)
        assert np.intersect1d(evaluation_indices, calibration_indices).size == 0
        predictions = logits_array[evaluation_indices].argmax(axis=1)
        accuracy = float(np.mean(predictions == labels_array[evaluation_indices]))
        split_accuracies.append({
            "seed": split_seed,
            "evaluation_examples": int(evaluation_indices.size),
            "top1_accuracy": accuracy,
        })
    print("Clean-test accuracy (computed only after the model is fixed):")
    for result in split_accuracies:
        print(
            f"  seed {result['seed']}: "
            f"{result['top1_accuracy']:.4f} on {result['evaluation_examples']:,} examples"
        )
    accuracy_path = RESULTS_DIR / "clean_test_accuracy_by_split.csv"
    with accuracy_path.open("w", newline="", encoding="utf-8") as output_file:
        writer = csv.DictWriter(
            output_file, fieldnames=["seed", "evaluation_examples", "top1_accuracy"]
        )
        writer.writeheader()
        writer.writerows(split_accuracies)
    return split_accuracies

if RUN_FULL_TRAINING:
    final_model, final_checkpoint_path = train_full_model()
    if torch.load(final_checkpoint_path, map_location="cpu", weights_only=False)["epoch"] == TRAINING_CONFIG["epochs"]:
        clean_test_accuracy_by_split = cache_and_evaluate_clean_test(
            final_model, final_checkpoint_path
        )
else:
    print("75-epoch training is disabled. No checkpoint or clean-test inference was run.")

75-epoch training is disabled. No checkpoint or clean-test inference was run.


## Recorded local validation

The saved cells record model-shape checks, finite losses and gradients, parameter updates during smoke training, the 128-example overfit check, and the one-epoch runtime probe. The temporary probe model is separate from the official trained classifiers. No calibration/evaluation split is used to fit the model or choose a checkpoint.